In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [2]:
import os 
print(os.getcwd())

d:\BookRecommendation\notebook


In [3]:
ratings_df = pd.read_csv(r'd:\BookRecommendation\data\processed\clean_book_ratings.csv')

In [4]:
ratings_df

,index,ISBN,Book_Title,Book_Author,Year_Of_Publication,Publisher,User_ID,Book_Rating,Age_Group,Country
0,30,0399135782,The Kitchen God's Wife,Amy Tan,1991,Putnam Pub Group,8,0,Unknown,canada
1,31,0399135782,The Kitchen God's Wife,Amy Tan,1991,Putnam Pub Group,11676,9,Unknown,Unknown
2,32,0399135782,The Kitchen God's Wife,Amy Tan,1991,Putnam Pub Group,29526,9,Adult,usa
3,33,0399135782,The Kitchen God's Wife,Amy Tan,1991,Putnam Pub Group,36836,0,Senior,usa
4,34,0399135782,The Kitchen God's Wife,Amy Tan,1991,Putnam Pub Group,46398,9,Middle_Age,usa
...,...,...,...,...,...,...,...,...,...,...
188377,1015621,0689823185,Where You Belong,Mary Ann McGuigan,1998,Simon Pulse,242299,0,Young_Adult,usa
188378,1016510,0553290703,Lightning,Patricia Potter,1992,Bantam Books,244685,9,Unknown,Unknown
188379,1016522,0345478940,Angel Falls,KRISTIN HANNAH,2004,Ballantine Books,244688,0,Adult,usa
188380,1016538,B00011SOYM,Grave Secrets,Kathy Reichs,2002,Scribner,244708,0,Unknown,united kingdom


In [5]:
print(f"Shape: {ratings_df.shape}")
print(f"\nColumns: {ratings_df.columns.tolist()}")
print(f"\nRating distribution:")
print(ratings_df['Book_Rating'].value_counts().sort_index())
print(f"\nExplicit ratings (>0): {(ratings_df['Book_Rating'] > 0).sum()}")
print(f"Implicit ratings (=0): {(ratings_df['Book_Rating'] == 0).sum()}")

Shape: (188382, 10)

Columns: ['index', 'ISBN', 'Book_Title', 'Book_Author', 'Year_Of_Publication', 'Publisher', 'User_ID', 'Book_Rating', 'Age_Group', 'Country']

Rating distribution:
Book_Rating
0     125037
1        226
2        341
3        703
4       1013
5       6001
6       4534
7      10251
8      15474
9      11782
10     13020
Name: count, dtype: int64

Explicit ratings (>0): 63345
Implicit ratings (=0): 125037


In [6]:
ratings_df['User_ID'].nunique()

7807

In [7]:
cf_ratings = ratings_df[ratings_df['Book_Rating'] > 0][
    ['User_ID', 'ISBN', 'Book_Rating']
].reset_index(drop=True)

print(f"Shape: {cf_ratings.shape}")
print(f"Unique users: {cf_ratings['User_ID'].nunique()}")
print(f"Unique books: {cf_ratings['ISBN'].nunique()}")

Shape: (63345, 3)
Unique users: 6874
Unique books: 4197


In [8]:
!pip install scikit-surprise

In [9]:
pip install "numpy<2" --upgrade

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install scikit-surprise --upgrade

Note: you may need to restart the kernel to use updated packages.


In [11]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy


In [12]:
reader = Reader(rating_scale=(1, 10))

# Load into Surprise format
data = Dataset.load_from_df(
    cf_ratings[['User_ID', 'ISBN', 'Book_Rating']], 
    reader
)

print("Data loaded into Surprise format successfully!")

Data loaded into Surprise format successfully!


In [13]:
data

In [14]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"Training ratings: {trainset.n_ratings}")
print(f"Testing ratings:  {len(testset)}")

Training ratings: 50676
Testing ratings:  12669


In [15]:
svd_model = SVD(n_factors=100, n_epochs=20, random_state=42)
svd_model.fit(trainset)

print("SVD model trained successfully!")

SVD model trained successfully!


In [16]:


# Make predictions on test set
predictions = svd_model.test(testset)

# Calculate RMSE and MAE
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")

RMSE: 1.5735
MAE:  1.2146
RMSE: 1.5735
MAE:  1.2146


In [18]:
users = pd.read_csv(r'd:\BookRecommendation\data\processed\user_clusters.csv')
book = pd.read_csv(r'd:\BookRecommendation\data\processed\book_clusters.csv')


In [21]:
users.columns

Index(['User_ID', 'user_avg_rating', 'user_rating_count', 'user_rating_std',
       'Age_Group', 'Country', 'favourite_author', 'old_books_ratio',
       'recent_books_ratio', 'genre_diversity', 'user_cluster'],
      dtype='str')

In [22]:
user_clusters = users.drop(columns=[ 'user_avg_rating', 'user_rating_count', 'user_rating_std',
       'Age_Group', 'Country', 'favourite_author', 'old_books_ratio',
       'recent_books_ratio', 'genre_diversity'])

In [24]:
book.columns

Index(['ISBN', 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std',
       'Year_Of_Publication', 'Publisher', 'Genre', 'Title_Author_combined',
       'Genre_grouped', 'book_cluster'],
      dtype='str')

In [25]:
book_clusters = book.drop(columns=[ 'Book_avg_rating', 'Book_rating_count', 'Book_rating_std',
       'Year_Of_Publication', 'Publisher', 'Genre', 'Title_Author_combined',
       'Genre_grouped'])

In [26]:
book_clusters

,ISBN,book_cluster
0,0399135782,4
1,0440234743,2
2,0452264464,2
3,0971880107,2
4,0345402871,2
...,...,...
5170,0689823185,1
5171,0553290703,3
5172,0345478940,1
5173,B00011SOYM,1


In [28]:
import pandas as pd
import numpy as np

# ── Load cluster files
# we did it above it the 
# book clusters 
# user clusters

# ── Merge ratings with cluster info
ratings_with_clusters = cf_ratings.merge(user_clusters, on='User_ID', how='left')
ratings_with_clusters = ratings_with_clusters.merge(book_clusters, on='ISBN', how='left')

# ── Find which book clusters each user cluster prefers
cluster_book_preferences = (ratings_with_clusters
    .groupby(['user_cluster', 'book_cluster'])
    .size()
    .reset_index(name='interaction_count')
    .sort_values(['user_cluster', 'interaction_count'], ascending=[True, False])
)

# ── Keep top 3 book clusters per user cluster
top_book_clusters = (cluster_book_preferences
    .groupby('user_cluster')
    .head(3)
    .reset_index(drop=True)
)

# ── All books list
all_books = book_clusters['ISBN'].tolist()

# ── Generate CF scores with hybrid candidate pool
MAX_CANDIDATES    = 200
CLUSTER_RATIO     = 0.70   # 70% from user's cluster
EXPLORE_RATIO     = 0.30   # 30% from other clusters

cluster_candidates_count = int(MAX_CANDIDATES * CLUSTER_RATIO)  # 140
explore_candidates_count = int(MAX_CANDIDATES * EXPLORE_RATIO)  # 60

cf_scores = []
all_users = user_clusters['User_ID'].unique()

for i, user_id in enumerate(all_users):

    # ── Get user's cluster
    user_cluster = user_clusters[
        user_clusters['User_ID'] == user_id
    ]['user_cluster'].values[0]

    # ── Get relevant book clusters for this user cluster
    relevant_book_clusters = top_book_clusters[
        top_book_clusters['user_cluster'] == user_cluster
    ]['book_cluster'].tolist()

    # ── Get books user already rated
    already_rated = set(cf_ratings[
        cf_ratings['User_ID'] == user_id
    ]['ISBN'].tolist())

    # ── CLUSTER CANDIDATES (70%)
    cluster_books = book_clusters[
        book_clusters['book_cluster'].isin(relevant_book_clusters)
    ]['ISBN'].tolist()

    cluster_books = [isbn for isbn in cluster_books
                    if isbn not in already_rated]

    np.random.shuffle(cluster_books)
    cluster_books = cluster_books[:cluster_candidates_count]

    # ── EXPLORATION CANDIDATES (30%)
    other_books = book_clusters[
        ~book_clusters['book_cluster'].isin(relevant_book_clusters)
    ]['ISBN'].tolist()

    other_books = [isbn for isbn in other_books
                  if isbn not in already_rated]

    np.random.shuffle(other_books)
    other_books = other_books[:explore_candidates_count]

    # ── Combine both candidate pools
    final_candidates = cluster_books + other_books

    # ── Generate CF scores
    for isbn in final_candidates:
        pred = svd_model.predict(uid=user_id, iid=isbn)
        cf_scores.append({
            'User_ID': user_id,
            'ISBN': isbn,
            'cf_predicted_score': pred.est
        })

    # ── Progress tracker
    if i % 500 == 0:
        print(f"Processed {i}/{len(all_users)} users...")

# ── Convert to dataframe
cf_scores_df = pd.DataFrame(cf_scores)
print(f"\nCF scores shape: {cf_scores_df.shape}")
print(cf_scores_df.head(10))

Processed 0/7807 users...
Processed 500/7807 users...
Processed 1000/7807 users...
Processed 1500/7807 users...
Processed 2000/7807 users...
Processed 2500/7807 users...
Processed 3000/7807 users...
Processed 3500/7807 users...
Processed 4000/7807 users...
Processed 4500/7807 users...
Processed 5000/7807 users...
Processed 5500/7807 users...
Processed 6000/7807 users...
Processed 6500/7807 users...
Processed 7000/7807 users...
Processed 7500/7807 users...

CF scores shape: (1561400, 3)
   User_ID        ISBN  cf_predicted_score
0        8  0380782413            7.783326
1        8  0375401601            7.537924
2        8  0440173701            7.709466
3        8  0449220184            7.502671
4        8  0618260552            7.913811
5        8  0375705856            7.767865
6        8  0449213730            7.691784
7        8  0440566827            7.630135
8        8  1573226521            7.895168
9        8  0425071790            7.782432


In [32]:
cf_scores_df.iloc[100000:100500]

,User_ID,ISBN,cf_predicted_score
100000,20409,044011585X,7.672991
100001,20409,0671744577,8.221985
100002,20409,0892965002,8.143653
100003,20409,0553238132,7.756110
100004,20409,0618260595,7.898146
...,...,...,...
100495,20462,0553279378,7.926272
100496,20462,0449220184,7.262732
100497,20462,0345250761,7.735518
100498,20462,0312963297,7.216254


In [30]:
import joblib

# Save CF scores
cf_scores_df.to_csv(r'd:\BookRecommendation\data\processed\cf_scores.csv', index=False)

# Save SVD model
joblib.dump(svd_model, r'd:\BookRecommendation\ml_models/cf_svd_model.pkl')

print("CF scores and SVD model saved!")

CF scores and SVD model saved!


In [33]:
# Find a user with many ratings
heavy_user = cf_ratings.groupby('User_ID').size().idxmax()
print(f"Most active user: {heavy_user}")
print(f"Ratings count: {cf_ratings[cf_ratings['User_ID'] == heavy_user].shape[0]}")

# Check their CF scores
print(cf_scores_df[cf_scores_df['User_ID'] == heavy_user]['cf_predicted_score'].describe())

Most active user: 11676
Ratings count: 1156
count    200.000000
mean       7.606138
std        0.536891
min        5.459711
25%        7.305998
50%        7.703733
75%        7.795111
max        9.220006
Name: cf_predicted_score, dtype: float64
